# AirbagsCV — Colab Training Notebook

This notebook trains the EfficientAD anomaly-detection model on the AITEX
dataset using **the repository's own scripts**. It does NOT reimplement
training logic. The repo is the source of truth.

**Before running:**
1. Upload AITEX to Google Drive at `/MyDrive/datasets/AITEX/` (must contain
   `NODefect_images/`, `Defect_images/`, `Mask_images/`).
2. Upload Imagenette (extracted from `imagenette2-160.tgz` at
   https://github.com/fastai/imagenette) to `/MyDrive/datasets/imagenette/`.
3. Select GPU runtime: `Runtime > Change runtime type > Hardware
   accelerator > GPU`.

See `notebooks/COLAB_GUIDE.md` in the repo for the full guide and
troubleshooting table.

## 1. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Runtime > Change runtime type > GPU, then Runtime > Restart session.')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
aitex_path = '/content/drive/MyDrive/datasets/AITEX'
if os.path.isdir(aitex_path):
    print('AITEX contents:', os.listdir(aitex_path))
else:
    print(f'WARNING: {aitex_path} does not exist. Upload AITEX first.')

## 3. Clone or update the repo

In [ ]:
%%bash
if [ -d /content/AirbagsCV ]; then
  cd /content/AirbagsCV
  git pull
else
  cd /content
  git clone https://github.com/MohamedAzzam4/AirbagsCV.git
fi

In [ ]:
%cd /content/AirbagsCV

## 4. Install dependencies

Colab ships with `torch` + `torchvision` preinstalled (CUDA build). Do NOT
reinstall torch from the default index — you will lose GPU.

In [ ]:
%%bash
cd /content/AirbagsCV
pip install -r requirements-colab.txt 2>&1 | tail -20

## 5. Define paths

In [ ]:
import os
os.environ['REPO_DIR']          = '/content/AirbagsCV'
os.environ['RAW_AITEX_DIR']     = '/content/drive/MyDrive/datasets/AITEX'
os.environ['IMAGENETTE_DIR']    = '/content/drive/MyDrive/datasets/imagenette/train'
os.environ['PREPARED_DATA_DIR'] = '/content/airbagcv_data/aitex_patches'
os.environ['RUNS_DIR']          = '/content/drive/MyDrive/AirbagsCV/runs'
os.environ['RESULTS_DIR']       = '/content/drive/MyDrive/AirbagsCV/results'

for d in ['PREPARED_DATA_DIR', 'RUNS_DIR', 'RESULTS_DIR']:
    os.makedirs(os.environ[d], exist_ok=True)

for k, v in os.environ.items():
    if k in ['REPO_DIR','RAW_AITEX_DIR','IMAGENETTE_DIR','PREPARED_DATA_DIR','RUNS_DIR','RESULTS_DIR']:
        print(f'{k:25s} {v}')

In [ ]:
# Verify dataset is reachable
required = ['NODefect_images', 'Defect_images', 'Mask_images']
for r in required:
    p = os.path.join(os.environ['RAW_AITEX_DIR'], r)
    ok = os.path.isdir(p)
    print(f"  {'OK ' if ok else 'MISSING'}  {p}")
    if not ok:
        raise FileNotFoundError(f'Missing {r}. Check your Drive path.')

# Verify imagenette
if not os.path.isdir(os.environ['IMAGENETTE_DIR']):
    raise FileNotFoundError(
        f"Missing imagenette at {os.environ['IMAGENETTE_DIR']}. "
        f"Download imagenette2-160.tgz from https://github.com/fastai/imagenette "
        f"and extract to /content/drive/MyDrive/datasets/imagenette/."
    )
print('All datasets reachable.')

## 6. Smoke test

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/smoke_test.py

## 7. Prepare AITEX data

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/prepare_aitex.py \
  --source "$RAW_AITEX_DIR" \
  --output "$PREPARED_DATA_DIR" \
  --patch-size 256 \
  --blank-threshold 0.5 \
  --seed 42 \
  --overwrite

## 8. Train EfficientAD

70 epochs on a T4 GPU takes ~30-60 minutes. The script will print the
checkpoint path at the end.

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/train_models.py \
  --model efficientad \
  --dataset aitex \
  --data-dir "$PREPARED_DATA_DIR" \
  --output-dir "$RUNS_DIR" \
  --epochs 70 \
  --batch-size 1 \
  --num-workers 2 \
  --accelerator gpu \
  --devices 1 \
  --precision 16-mixed \
  --seed 42 \
  --imagenet-dir "$IMAGENETTE_DIR"

## 9. Resume training (only if Colab disconnected mid-run)

Skip this cell if step 8 finished successfully.

In [ ]:
import glob, os
run_dir = os.path.join(os.environ['RUNS_DIR'], 'efficientad_aitex')
ckpts = sorted(glob.glob(f'{run_dir}/**/*.ckpt', recursive=True),
               key=lambda p: os.path.getmtime(p))
if ckpts:
    CKPT = ckpts[-1]
    print('Latest checkpoint:', CKPT)
else:
    raise FileNotFoundError(f'No checkpoint found under {run_dir}. Did training run?')

In [ ]:
%%bash -s "$CKPT"
cd /content/AirbagsCV
python scripts/train_models.py \
  --model efficientad \
  --dataset aitex \
  --data-dir "$PREPARED_DATA_DIR" \
  --output-dir "$RUNS_DIR" \
  --epochs 70 \
  --batch-size 1 \
  --num-workers 2 \
  --accelerator gpu \
  --devices 1 \
  --precision 16-mixed \
  --seed 42 \
  --imagenet-dir "$IMAGENETTE_DIR" \
  --resume-from-checkpoint "$1"

## 10. Run honest inference benchmark

Measures real per-image latency with warmup + CUDA sync. Reports p50/p95/p99.

In [ ]:
import glob, os
run_dir = os.path.join(os.environ['RUNS_DIR'], 'efficientad_aitex')
ckpts = sorted(glob.glob(f'{run_dir}/**/*.ckpt', recursive=True),
               key=lambda p: os.path.getmtime(p))
os.environ['CHECKPOINT_PATH'] = ckpts[-1] if ckpts else \
    'results/efficientad_aitex/EfficientAd/aitex/latest/weights/lightning/model.ckpt'
print('CHECKPOINT_PATH =', os.environ['CHECKPOINT_PATH'])

In [ ]:
%%bash
cd /content/AirbagsCV
python scripts/benchmark_inference.py \
  --checkpoint "$CHECKPOINT_PATH" \
  --model efficientad \
  --data-dir "$PREPARED_DATA_DIR" \
  --output-csv "$RESULTS_DIR/latency.csv" \
  --device cuda \
  --warmup 20 \
  --iterations 200

## 11. Verify outputs are on Drive

In [ ]:
import os
for label, path in [('Runs dir', os.environ['RUNS_DIR']),
                    ('Results dir', os.environ['RESULTS_DIR'])]:
    print(f'\n=== {label}: {path} ===')
    n = 0
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, '').count(os.sep)
        if depth > 3:
            continue
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:5]:
            print(f"{indent}  {f}")
            n += 1
        if len(files) > 5:
            print(f"{indent}  ... and {len(files)-5} more")
    print(f'Total visible files: {n}')

## 12. Zip outputs (optional)

In [ ]:
%%bash
cd /content
zip -r /content/drive/MyDrive/AirbagsCV/airbagcv_run_outputs.zip \
  "$RUNS_DIR" \
  "$RESULTS_DIR" 2>&1 | tail -5
ls -lh /content/drive/MyDrive/AirbagsCV/airbagcv_run_outputs.zip

## Done

Your trained checkpoint, metrics, and benchmark are on Google Drive at
`/MyDrive/AirbagsCV/`. To run the interactive demo locally:

1. Download your checkpoint from Drive.
2. Clone the repo on your local machine.
3. Place the checkpoint under
   `results/efficientad_aitex/EfficientAd/aitex/latest/weights/lightning/model.ckpt`.
4. Run `python demo/app.py` and open http://127.0.0.1:7860/.

See `notebooks/COLAB_GUIDE.md` for the troubleshooting table.